In [225]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.tsa.stattools import pacf
import numpy as np

In [226]:
import os
import sys
current_dir = os.path.dirname(os.path.realpath(__name__))
parent_dir = os.path.dirname(current_dir)
print(parent_dir)
sys.path.append(parent_dir)
from src.data.data_loader import DataLoader
dl = DataLoader()
dl.prepare_df()
prepared_df = dl.prepared_df
dl.train_test_split()

C:\Users\jonah.eisen\dsi\global-stock-index-ml-classification


In [227]:
train_pacf = pacf(dl.y_train.rolling(window=7).mean().dropna(), method='ywm')
alpha = 0.1
ar_terms = [i for i,val in enumerate(np.abs(train_pacf) > alpha) if val]
ar_terms = ar_terms[1:]
ar_terms

[1, 2, 3, 4, 6, 7, 8, 9, 11, 14, 15, 16, 21, 22, 23, 29]

In [228]:
prepared_df.head()

,Last Close,Close,Last Adj Close,Volume,LowProportion,HighProportion,Delta Close
Date,,,,,,,
2003-01-10,5210.040039,5209.209961,5210.339844,1.477200e+09,0.991511,1.004248,-0.830078
2003-01-13,5209.799805,5233.660156,5209.799805,1.371500e+09,0.995975,1.006883,23.860351
2003-01-14,5209.209961,5171.450195,5209.209961,1.355600e+09,0.996499,1.004696,-37.759766
2003-01-15,5233.729980,5165.339844,5233.660156,1.385200e+09,0.986618,1.000290,-68.390136
2003-01-16,5171.450195,5108.509766,5171.450195,1.534600e+09,0.996568,1.006555,-62.940429


In [229]:
p = max(ar_terms)
lag_dict = {}
for j in ar_terms:
    lag_dict[f'y_(t-{j})'] = []
for i in range(p,prepared_df.shape[0]):
    for j in ar_terms:
        lag_val = prepared_df['Delta Close'].iloc[i-j]
        lag_dict[f'y_(t-{j})'].append(lag_val) 

In [230]:
cropped_df = prepared_df.iloc[p:,:]
cropped_df.shape

(4598, 7)

In [231]:
cropped_df = cropped_df.assign(**lag_dict)
cropped_df.shape

(4598, 23)

In [232]:
cropped_df  = cropped_df.rolling(window=7, center=False).mean().dropna()

In [233]:
cropped_df.head()

,Last Close,Close,Last Adj Close,Volume,LowProportion,HighProportion,Delta Close,y_(t-1),y_(t-2),y_(t-3),...,y_(t-8),y_(t-9),y_(t-11),y_(t-14),y_(t-15),y_(t-16),y_(t-21),y_(t-22),y_(t-23),y_(t-29)
Date,,,,,,,,,,,,,,,,,,,,,
2003-03-04,4711.395717,4685.448451,4711.389927,1.302586e+09,0.990314,1.005602,-25.947266,-24.372977,-9.840123,-22.803013,...,10.932826,6.837123,-4.241420,-56.441546,-30.238700,-29.140067,-29.309989,-46.592773,-67.771345,-62.949986
2003-03-05,4690.092843,4672.968471,4690.087054,1.320700e+09,0.992351,1.006698,-17.124372,-25.947266,-24.372977,-9.840123,...,19.279994,10.932826,14.029994,-58.044364,-56.441546,-30.238700,-23.800014,-29.309989,-46.592773,-65.777065
2003-03-06,4685.454241,4671.699916,4685.448451,1.291929e+09,0.993056,1.006399,-13.754325,-17.124372,-25.947266,-24.372977,...,16.658552,19.279994,6.837123,-35.620047,-58.044364,-56.441546,-4.975725,-23.800014,-29.309989,-81.012835
2003-03-07,4672.975656,4648.205636,4672.968471,1.289557e+09,0.993423,1.007053,-24.770019,-13.754325,-17.124372,-25.947266,...,-2.238630,16.658552,10.932826,-4.241420,-35.620047,-58.044364,-8.971540,-4.975725,-23.800014,-105.432896
2003-03-10,4671.704241,4617.287110,4671.699916,1.277171e+09,0.989954,1.005261,-54.417132,-24.770019,-13.754325,-17.124372,...,-26.855818,-2.238630,19.279994,14.029994,-4.241420,-35.620047,-33.711496,-8.971540,-4.975725,-101.262835


In [234]:
delta_close = cropped_df['Delta Close']
y = DataLoader.quantize_delta_close(delta_close)
cropped_df.drop(columns=['Last Close', 'Close', 'Delta Close', 'Last Adj Close'],inplace=True)

In [235]:
cropped_df.head()

,Volume,LowProportion,HighProportion,y_(t-1),y_(t-2),y_(t-3),y_(t-4),y_(t-6),y_(t-7),y_(t-8),y_(t-9),y_(t-11),y_(t-14),y_(t-15),y_(t-16),y_(t-21),y_(t-22),y_(t-23),y_(t-29)
Date,,,,,,,,,,,,,,,,,,,
2003-03-04,1.302586e+09,0.990314,1.005602,-24.372977,-9.840123,-22.803013,-26.855818,16.658552,19.279994,10.932826,6.837123,-4.241420,-56.441546,-30.238700,-29.140067,-29.309989,-46.592773,-67.771345,-62.949986
2003-03-05,1.320700e+09,0.992351,1.006698,-25.947266,-24.372977,-9.840123,-22.803013,-2.238630,16.658552,19.279994,10.932826,14.029994,-58.044364,-56.441546,-30.238700,-23.800014,-29.309989,-46.592773,-65.777065
2003-03-06,1.291929e+09,0.993056,1.006399,-17.124372,-25.947266,-24.372977,-9.840123,-26.855818,-2.238630,16.658552,19.279994,6.837123,-35.620047,-58.044364,-56.441546,-4.975725,-23.800014,-29.309989,-81.012835
2003-03-07,1.289557e+09,0.993423,1.007053,-13.754325,-17.124372,-25.947266,-24.372977,-22.803013,-26.855818,-2.238630,16.658552,10.932826,-4.241420,-35.620047,-58.044364,-8.971540,-4.975725,-23.800014,-105.432896
2003-03-10,1.277171e+09,0.989954,1.005261,-24.770019,-13.754325,-17.124372,-25.947266,-9.840123,-22.803013,-26.855818,-2.238630,19.279994,14.029994,-4.241420,-35.620047,-33.711496,-8.971540,-4.975725,-101.262835


In [236]:
len(lag_dict.keys())

16

In [237]:
val_size = 0.2
X_train, X_test = DataLoader.time_split_2D(cropped_df)
X_train, X_val = DataLoader.time_split_2D(X_train)
y_train, y_test = DataLoader.time_split_1D(y)
y_train, y_val = DataLoader.time_split_1D(y_train)

In [238]:
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
y_prob = rf.predict_proba(X_val)[:,1]



In [239]:
print("Random Forest Accuracy:", rf.score(X_val, y_val))
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

Random Forest Accuracy: 0.8680272108843538
              precision    recall  f1-score   support

           0       0.87      0.83      0.85       321
           1       0.87      0.90      0.88       414

    accuracy                           0.87       735
   macro avg       0.87      0.86      0.87       735
weighted avg       0.87      0.87      0.87       735

ROC-AUC: 0.9400198654566798
